In [2]:
# =========================
# 0. 环境与依赖
# =========================
# !pip install -q fastparquet

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.manifold import TSNE

import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed
)

warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

/public/home/songling/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.3)
  from scipy.sparse import csr_matrix, issparse


DEVICE = cuda


In [5]:
MODEL_CHECKPOINT = "microsoft/deberta-v3-small"
print("MODEL_CHECKPOINT =", MODEL_CHECKPOINT)
print("将从 Hugging Face 在线拉取模型与 tokenizer（首次运行会下载缓存）")


True
True
True


In [8]:
from pathlib import Path

# 你项目根目录，可按需修改
SEARCH_ROOT = Path("/public/home/songling/LLM-Detect AI Generated Text")

targets = {
    "pile2": "pile2.parquet",
    "pile3": "plies3.parquet",
    "pile4": "plies4.parquet",
    "Ultra": "Ultra.parquet",
    "lmsys": "lmsys.parquet",
    "Human_LLM": "data.parquet",
    "valid": "nonTargetText_llm_slightly_modified_gen.csv",
}

found_paths = {}

for key, filename in targets.items():
    matches = list(SEARCH_ROOT.rglob(filename))
    found_paths[key] = [str(p) for p in matches]

for key, paths in found_paths.items():
    print(f"\n[{key}]")
    if paths:
        for i, p in enumerate(paths, 1):
            print(f"  {i}. {p}")
    else:
        print("  未找到")


[pile2]
  1. /public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/pile2.parquet

[pile3]
  1. /public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/plies3.parquet

[pile4]
  1. /public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/plies4.parquet

[Ultra]
  1. /public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/Ultra.parquet

[lmsys]
  1. /public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/lmsys.parquet

[Human_LLM]
  1. /public/home/songling/LLM-Detect AI Generated Text/human-vs-llm-text-corpus/data.parquet

[valid]
  1. /public/home/songling/LLM-Detect AI Generated Text/code/nonTargetText_llm_slightly_modified_gen.csv


In [9]:
from pathlib import Path

# =========================
# 1. 全局配置
# =========================
MODEL_CHECKPOINT = "microsoft/deberta-v3-small"

OUTPUT_DIR = Path("./deberta_v3_small_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_GROUPS = {
    "compare": True,      # 表5-7
    "hparam": True,       # 表5-8
    "multisource": True,  # 表5-9
    "vis": True,          # 图5-2/3/4/5/6/7/8/9
    "error": True         # 表5-14/15
}

QUICK_MODE = False   # True先通流程，False完整跑

EPOCHS_DEFAULT = 2 if QUICK_MODE else 4
EARLY_STOPPING_PATIENCE = 2 if QUICK_MODE else 5
MAX_LENGTH_DEFAULT = 384
BATCH_SIZE_DEFAULT = 32 if QUICK_MODE else 64
LR_DEFAULT = 1e-4
GRAD_ACCUM_DEFAULT = 1

# DDP配置：按4卡4090训练
DDP_TARGET_GPUS = 4
BF16_ENABLED = torch.cuda.is_available() and torch.cuda.is_bf16_supported()


GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
if GPU_COUNT > 0:
    print(f"GPU_COUNT={GPU_COUNT}, WORLD_SIZE={WORLD_SIZE}")

MAX_TSNE_SAMPLES = 1000 if QUICK_MODE else 1600

DATA_PATHS = {
    "pile2": "/public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/pile2.parquet",
    "pile3": "/public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/plies3.parquet",
    "pile4": "/public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/plies4.parquet",
    "Ultra": "/public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/Ultra.parquet",
    "lmsys": "/public/home/songling/LLM-Detect AI Generated Text/plies-and-ultra/lmsys.parquet",
    "Human_LLM": "/public/home/songling/LLM-Detect AI Generated Text/human-vs-llm-text-corpus/data.parquet",
    "valid": "/public/home/songling/LLM-Detect AI Generated Text/code/nonTargetText_llm_slightly_modified_gen.csv",
}

In [ ]:
# =========================
# DDP 启动说明（4x4090）
# 在服务器终端运行，不在 notebook 单元里运行
# =========================
print("推荐命令：")
print("CUDA_VISIBLE_DEVICES=0,1,2,3 torchrun --nproc_per_node=4 --master_port=29600 run_deberata.py")
print("说明：先将本notebook导出为脚本 run_deberata.py，再用 torchrun 启动。")


In [10]:
# =========================
# 2. 工具函数
# =========================
def clean_text_series(s):
    return (
        s.fillna("")
         .astype(str)
         .str.replace(r"\r", " ", regex=True)
         .str.replace(r"\n+", " ", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )

def dedup_df(df):
    return df.drop_duplicates(subset=["text"]).reset_index(drop=True)

def normalize_binary_label(x):
    x = int(x)
    if x not in [0, 1]:
        raise ValueError(f"发现非法标签: {x}")
    return x

def dataset_stats(name, df):
    txt = df["text"].astype(str)
    return {
        "数据集": name,
        "样本数": len(df),
        "label0_human": int((df["label"] == 0).sum()),
        "label1_ai": int((df["label"] == 1).sum()),
        "平均长度": round(txt.str.len().mean(), 2),
        "最大长度": int(txt.str.len().max()),
        "最小长度": int(txt.str.len().min()),
        "平均词数": round(txt.str.split().apply(len).mean(), 2),
    }

def split_train_valid(df, test_size=0.1, seed=SEED):
    train_df, valid_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"]
    )
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True)

def balance_sample(df, seed=SEED):
    c0 = df[df["label"] == 0]
    c1 = df[df["label"] == 1]
    n = min(len(c0), len(c1))
    out = pd.concat([
        c0.sample(n=n, random_state=seed),
        c1.sample(n=n, random_state=seed)
    ], axis=0).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

def type_token_ratio(text):
    toks = str(text).split()
    if len(toks) == 0:
        return 0.0
    return len(set(toks)) / len(toks)

def length_bucket_by_words(text):
    n = len(str(text).split())
    if n <= 100:
        return "0-100"
    elif n <= 200:
        return "101-200"
    elif n <= 400:
        return "201-400"
    else:
        return "401+"

In [ ]:
# =========================
# 3. 读取数据
# 统一标签：0=Human, 1=AI
# =========================
pile2 = pd.read_parquet(DATA_PATHS["pile2"], engine="fastparquet")
pile3 = pd.read_parquet(DATA_PATHS["pile3"], engine="fastparquet")
pile4 = pd.read_parquet(DATA_PATHS["pile4"], engine="fastparquet")
Ultra = pd.read_parquet(DATA_PATHS["Ultra"], engine="fastparquet")
lmsys = pd.read_parquet(DATA_PATHS["lmsys"], engine="fastparquet")

Human_LLM = pd.read_parquet(DATA_PATHS["Human_LLM"], engine="fastparquet")
Human_LLM["label"] = np.where(Human_LLM["source"] == "Human", 0, 1)
Human_LLM = Human_LLM[["text", "label"]]

valid_official = pd.read_csv(DATA_PATHS["valid"])
assert "text" in valid_official.columns and "label" in valid_official.columns, "valid集必须包含 text 和 label 列"

def prep_df(df, name):
    assert "text" in df.columns and "label" in df.columns, f"{name} 缺少 text/label"
    out = df[["text", "label"]].copy()
    out["text"] = clean_text_series(out["text"])
    out["label"] = out["label"].apply(normalize_binary_label)
    out = out[out["text"].str.len() > 0].reset_index(drop=True)
    return out

pile2 = prep_df(pile2, "pile2")
pile3 = prep_df(pile3, "pile3")
pile4 = prep_df(pile4, "pile4")
Ultra = prep_df(Ultra, "Ultra")
lmsys = prep_df(lmsys, "lmsys")
Human_LLM = prep_df(Human_LLM, "Human_LLM")
valid_official = prep_df(valid_official, "valid_official")

source_stats = pd.DataFrame([
    dataset_stats("pile2", pile2),
    dataset_stats("pile3", pile3),
    dataset_stats("pile4", pile4),
    dataset_stats("Ultra", Ultra),
    dataset_stats("lmsys", lmsys),
    dataset_stats("Human_LLM", Human_LLM),
    dataset_stats("valid_official", valid_official),
])
source_stats.to_csv(OUTPUT_DIR / "all_source_stats.csv", index=False)
source_stats

In [ ]:
# =========================
# 4. 构造“官方 / 外部A / 外部B”
# 你这套 DeBERTa 代码里没有单独的官方train，所以这里做如下映射：
# official_data = Human_LLM
# external_A   = pile2 + pile3 + pile4
# external_B   = Ultra + lmsys
# valid集固定用 nonTargetText_llm_slightly_modified_gen.csv
# =========================
official_data = dedup_df(Human_LLM.copy())
external_A = dedup_df(pd.concat([pile2, pile3, pile4], axis=0).reset_index(drop=True))
external_B = dedup_df(pd.concat([Ultra, lmsys], axis=0).reset_index(drop=True))

group_stats = pd.DataFrame([
    dataset_stats("official_data", official_data),
    dataset_stats("external_A", external_A),
    dataset_stats("external_B", external_B),
])
group_stats.to_csv(OUTPUT_DIR / "dataset_group_stats.csv", index=False)
group_stats

In [ ]:
# =========================
# 5. 构造实验数据方案
# valid固定使用官方验证集
# =========================
def build_dataset_variant(
    name,
    use_official=True,
    use_external_A=False,
    use_external_B=False,
    dedup=True,
    balance=False
):
    parts = []
    if use_official:
        parts.append(official_data)
    if use_external_A:
        parts.append(external_A)
    if use_external_B:
        parts.append(external_B)

    df = pd.concat(parts, axis=0).reset_index(drop=True)
    df["text"] = clean_text_series(df["text"])
    df = df[df["text"].str.len() > 0].reset_index(drop=True)

    if dedup:
        df = dedup_df(df)
    if balance:
        df = balance_sample(df)

    train_df = df.reset_index(drop=True)
    valid_df = valid_official.copy().reset_index(drop=True)

    return {
        "name": name,
        "full_df": df,
        "train_df": train_df,
        "valid_df": valid_df
    }

base_variant = build_dataset_variant(
    name="official_externalAB",
    use_official=True,
    use_external_A=True,
    use_external_B=True,
    dedup=True,
    balance=False
)

print(base_variant["train_df"].shape, base_variant["valid_df"].shape)

In [ ]:
# =========================
# 6. Tokenizer / Metrics / Callback
# =========================
def build_tokenizer(model_checkpoint):
    model_checkpoint = str(model_checkpoint)
    try:
        # DeBERTa-v3 在部分环境 fast tokenizer 可能触发 tiktoken 依赖，优先 slow tokenizer
        return AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)
    except Exception as e:
        raise RuntimeError(
            f"Tokenizer 加载失败: {e}. 建议先安装 sentencepiece: pip install sentencepiece"
        )

tokenizer = build_tokenizer(MODEL_CHECKPOINT)

def tokenize_function(examples, max_length=MAX_LENGTH_DEFAULT):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    roc = roc_auc_score(labels, probs)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc
    }

class HistoryCallback(TrainerCallback):
    def __init__(self):
        self.logs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            item = dict(logs)
            item["step"] = state.global_step
            item["epoch"] = state.epoch
            self.logs.append(item)


In [ ]:
# =========================
# 7. 训练函数（DDP 4卡）
# =========================
def train_one_experiment(
    exp_name,
    train_df,
    valid_df,
    model_checkpoint=MODEL_CHECKPOINT,
    max_length=MAX_LENGTH_DEFAULT,
    learning_rate=LR_DEFAULT,
    batch_size=BATCH_SIZE_DEFAULT,
    grad_accum=GRAD_ACCUM_DEFAULT,
    num_epochs=EPOCHS_DEFAULT,
    weight_decay=0.01,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    fp16=True
):
    exp_dir = OUTPUT_DIR / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    visible_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "-1"))
    ddp_enabled = world_size > 1

    if ddp_enabled and world_size != DDP_TARGET_GPUS and local_rank in (-1, 0):
        print(f"[WARN] 当前WORLD_SIZE={world_size}，目标是{DDP_TARGET_GPUS}卡。")
    if (not ddp_enabled) and visible_gpus >= DDP_TARGET_GPUS:
        print("[WARN] 当前不是DDP进程（WORLD_SIZE=1）。若要4卡DDP，请用 torchrun 启动。")

    effective_world_size = world_size if ddp_enabled else 1
    per_device_batch = max(1, batch_size // effective_world_size)
    if batch_size % effective_world_size != 0 and local_rank in (-1, 0):
        print(f"[WARN] batch_size={batch_size} 不能整除WORLD_SIZE={effective_world_size}，每卡batch使用 {per_device_batch}。")

    use_bf16 = BF16_ENABLED
    use_fp16 = (not use_bf16) and fp16 and torch.cuda.is_available()

    ds_train = Dataset.from_pandas(
        train_df[["text", "label"]].reset_index(drop=True),
        preserve_index=False
    )
    ds_valid = Dataset.from_pandas(
        valid_df[["text", "label"]].reset_index(drop=True),
        preserve_index=False
    )

    ds_train = ds_train.map(
        lambda x: tokenize_function(x, max_length=max_length),
        batched=True,
        load_from_cache_file=True
    )
    ds_valid = ds_valid.map(
        lambda x: tokenize_function(x, max_length=max_length),
        batched=True,
        load_from_cache_file=True
    )

    if "text" in ds_train.column_names:
        ds_train = ds_train.remove_columns(["text"])
    if "text" in ds_valid.column_names:
        ds_valid = ds_valid.remove_columns(["text"])

    ds_train.set_format("torch")
    ds_valid.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=2
    )

    history_cb = HistoryCallback()
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=early_stopping_patience
    )

    args = TrainingArguments(
        output_dir=str(exp_dir / "model"),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        bf16=use_bf16,
        fp16=use_fp16,
        tf32=True,
        optim="adamw_torch",
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        dataloader_num_workers=4,
        dataloader_pin_memory=torch.cuda.is_available(),
        ddp_backend="nccl" if ddp_enabled else None,
        ddp_find_unused_parameters=False if ddp_enabled else None,
        save_on_each_node=False,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model="roc_auc",
        report_to="none",
        save_total_limit=3,
        seed=SEED,
        disable_tqdm=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_valid,
        tokenizer=tokenizer,
        callbacks=[history_cb, early_stopping],
        compute_metrics=compute_metrics
    )

    if local_rank in (-1, 0):
        print(f"[START] {exp_name}")
        print(
            f"train={len(train_df)}, valid={len(valid_df)}, max_length={max_length}, lr={learning_rate}, "
            f"global_batch={batch_size}, per_device_batch={per_device_batch}, world_size={effective_world_size}, "
            f"ddp_enabled={ddp_enabled}, bf16={use_bf16}, fp16={use_fp16}, epochs={num_epochs}"
        )

    trainer.train()

    pred_output = trainer.predict(ds_valid)
    logits = pred_output.predictions
    labels = pred_output.label_ids

    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    roc = roc_auc_score(labels, probs)

    result = {
        "exp_name": exp_name,
        "train_size": len(train_df),
        "valid_size": len(valid_df),
        "max_length": max_length,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "grad_accum": grad_accum,
        "epochs": num_epochs,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc
    }

    metrics_df = pd.DataFrame([result])
    metrics_df.to_csv(exp_dir / "metrics_summary.csv", index=False)

    pred_df = pd.DataFrame({
        "text": valid_df["text"].values,
        "label": labels,
        "prob_ai": probs,
        "pred": preds
    })
    pred_df.to_csv(exp_dir / "valid_predictions.csv", index=False)

    history_df = pd.DataFrame(history_cb.logs)
    history_df.to_csv(exp_dir / "train_history.csv", index=False)

    with open(exp_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump({
            "model_checkpoint": str(model_checkpoint),
            "max_length": max_length,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "grad_accum": grad_accum,
            "num_epochs": num_epochs,
            "weight_decay": weight_decay,
            "early_stopping_patience": early_stopping_patience,
            "fp16": fp16,
            "bf16": use_bf16,
            "world_size": effective_world_size
        }, f, ensure_ascii=False, indent=2)

    return result, labels, probs, preds, history_df


In [ ]:
# =========================
# 8. 画图函数
# =========================
def plot_loss_auc_from_history(history_logs, exp_name):
    df = pd.DataFrame(history_logs)
    if len(df) == 0:
        return

    exp_dir = OUTPUT_DIR / exp_name

    plt.figure(figsize=(8, 5))
    if "loss" in df.columns:
        d1 = df.dropna(subset=["loss"])
        if len(d1) > 0:
            plt.plot(d1["step"], d1["loss"], label="train_loss")
    if "eval_loss" in df.columns:
        d2 = df.dropna(subset=["eval_loss"])
        if len(d2) > 0:
            plt.plot(d2["step"], d2["eval_loss"], label="valid_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"{exp_name} Loss Curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_dir / "loss_curve.png", dpi=200)
    plt.show()

    if "eval_roc_auc" in df.columns:
        d3 = df.dropna(subset=["eval_roc_auc"])
        if len(d3) > 0:
            plt.figure(figsize=(8, 5))
            plt.plot(d3["epoch"], d3["eval_roc_auc"], marker="o")
            plt.xlabel("Epoch")
            plt.ylabel("Validation ROC-AUC")
            plt.title(f"{exp_name} Validation ROC-AUC")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(exp_dir / "valid_auc_curve.png", dpi=200)
            plt.show()

def plot_roc(labels, probs, exp_name):
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_val = roc_auc_score(labels, probs)

    exp_dir = OUTPUT_DIR / exp_name
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_val:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{exp_name} ROC Curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_dir / "roc_curve.png", dpi=200)
    plt.show()

def plot_pr_curve(labels, probs, exp_name):
    p, r, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(r, p)
    exp_dir = OUTPUT_DIR / exp_name

    plt.figure(figsize=(6, 6))
    plt.plot(r, p, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{exp_name} Precision-Recall Curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_dir / "pr_curve.png", dpi=200)
    plt.show()

def plot_confmat(labels, preds, exp_name):
    cm = confusion_matrix(labels, preds)
    exp_dir = OUTPUT_DIR / exp_name

    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"])
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title(f"{exp_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix.png", dpi=200)
    plt.show()

In [ ]:
# =========================
# 9. 表5-7：DeBERTa 结果对比
# official / official+A / official+A+B
# =========================
compare_results = []

if RUN_GROUPS["compare"]:
    compare_variants = [
        {
            "name": "deberta_official_only",
            "dataset": build_dataset_variant(
                name="official_only",
                use_official=True,
                use_external_A=False,
                use_external_B=False,
                dedup=True,
                balance=False
            ),
            "max_length": 384,
            "learning_rate": 1e-4,
            "batch_size": 64 if not QUICK_MODE else 32,
            "epochs": EPOCHS_DEFAULT
        },
        {
            "name": "deberta_official_externalA",
            "dataset": build_dataset_variant(
                name="official_externalA",
                use_official=True,
                use_external_A=True,
                use_external_B=False,
                dedup=True,
                balance=False
            ),
            "max_length": 384,
            "learning_rate": 1e-4,
            "batch_size": 64 if not QUICK_MODE else 32,
            "epochs": EPOCHS_DEFAULT
        },
        {
            "name": "deberta_official_externalAB",
            "dataset": build_dataset_variant(
                name="official_externalAB",
                use_official=True,
                use_external_A=True,
                use_external_B=True,
                dedup=True,
                balance=False
            ),
            "max_length": 384,
            "learning_rate": 1e-4,
            "batch_size": 64 if not QUICK_MODE else 32,
            "epochs": EPOCHS_DEFAULT
        },
    ]

    for cfg in compare_variants:
        print("=" * 100)
        print("RUN:", cfg["name"])

        res, labels, probs, preds, hist = train_one_experiment(
            exp_name=cfg["name"],
            train_df=cfg["dataset"]["train_df"],
            valid_df=cfg["dataset"]["valid_df"],
            max_length=cfg["max_length"],
            learning_rate=cfg["learning_rate"],
            batch_size=cfg["batch_size"],
            num_epochs=cfg["epochs"],
            fp16=True
        )

        compare_results.append({
            "模型": "DeBERTa-v3-small",
            "训练数据": cfg["dataset"]["name"],
            "max_length": cfg["max_length"],
            "Accuracy": round(res["accuracy"], 6),
            "Precision": round(res["precision"], 6),
            "Recall": round(res["recall"], 6),
            "F1-score": round(res["f1"], 6),
            "ROC-AUC": round(res["roc_auc"], 6)
        })

    table57_deberta = pd.DataFrame(compare_results)
    table57_deberta.to_csv(OUTPUT_DIR / "table_5_7_deberta_compare.csv", index=False)
    print(table57_deberta.to_string(index=False))

In [ ]:
# =========================
# 10. 表5-8：超参数实验
# =========================
hparam_results = []

if RUN_GROUPS["hparam"]:
    hparam_dataset = build_dataset_variant(
        name="official_externalAB_for_hparam",
        use_official=True,
        use_external_A=True,
        use_external_B=True,
        dedup=True,
        balance=False
    )

    hparam_grid = [
        {"exp_id": "E1", "lr": 2e-5, "batch": 16, "epochs": 3, "max_length": 256},
        {"exp_id": "E2", "lr": 2e-5, "batch": 16, "epochs": 5, "max_length": 256},
        {"exp_id": "E3", "lr": 3e-5, "batch": 16, "epochs": 3, "max_length": 512},
        {"exp_id": "E4", "lr": 5e-5, "batch": 8,  "epochs": 3, "max_length": 512},
    ]

    for cfg in hparam_grid:
        exp_name = f"deberta_hparam_{cfg['exp_id']}"

        print("=" * 100)
        print("RUN:", exp_name)

        res, labels, probs, preds, hist = train_one_experiment(
            exp_name=exp_name,
            train_df=hparam_dataset["train_df"],
            valid_df=hparam_dataset["valid_df"],
            max_length=cfg["max_length"],
            learning_rate=cfg["lr"],
            batch_size=cfg["batch"],
            num_epochs=cfg["epochs"],
            fp16=True
        )

        hparam_results.append({
            "实验编号": cfg["exp_id"],
            "learning rate": cfg["lr"],
            "batch size": cfg["batch"],
            "epoch": cfg["epochs"],
            "max_length": cfg["max_length"],
            "early stopping": "是",
            "ROC-AUC": round(res["roc_auc"], 6),
            "Accuracy": round(res["accuracy"], 6),
            "Precision": round(res["precision"], 6),
            "Recall": round(res["recall"], 6),
            "F1-score": round(res["f1"], 6),
        })

    table58 = pd.DataFrame(hparam_results)
    table58.to_csv(OUTPUT_DIR / "table_5_8_deberta_hparams.csv", index=False)
    print(table58.to_string(index=False))

In [ ]:
# =========================
# 11. 表5-9：多源数据融合训练实验
# =========================
multisource_results = []

if RUN_GROUPS["multisource"]:
    variants = [
        {
            "name": "S1",
            "cfg": dict(use_official=True, use_external_A=False, use_external_B=False, dedup=True, balance=False)
        },
        {
            "name": "S2",
            "cfg": dict(use_official=True, use_external_A=True, use_external_B=False, dedup=True, balance=False)
        },
        {
            "name": "S3",
            "cfg": dict(use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=False)
        },
        {
            "name": "S4",
            "cfg": dict(use_official=True, use_external_A=True, use_external_B=True, dedup=True, balance=True)
        },
    ]

    for item in variants:
        ds = build_dataset_variant(name=item["name"], **item["cfg"])
        exp_name = f"deberta_multisource_{item['name']}"

        print("=" * 100)
        print("RUN:", exp_name)

        res, labels, probs, preds, hist = train_one_experiment(
            exp_name=exp_name,
            train_df=ds["train_df"],
            valid_df=ds["valid_df"],
            max_length=384,
            learning_rate=1e-4,
            batch_size=64 if not QUICK_MODE else 32,
            num_epochs=EPOCHS_DEFAULT,
            fp16=True
        )

        multisource_results.append({
            "训练数据方案": item["name"],
            "官方数据": "√" if item["cfg"]["use_official"] else "×",
            "外部数据A": "√" if item["cfg"]["use_external_A"] else "×",
            "外部数据B": "√" if item["cfg"]["use_external_B"] else "×",
            "去重": "√" if item["cfg"]["dedup"] else "×",
            "平衡采样": "√" if item["cfg"]["balance"] else "×",
            "ROC-AUC": round(res["roc_auc"], 6),
            "Accuracy": round(res["accuracy"], 6),
            "Precision": round(res["precision"], 6),
            "Recall": round(res["recall"], 6),
            "F1-score": round(res["f1"], 6),
        })

    table59 = pd.DataFrame(multisource_results)
    table59.to_csv(OUTPUT_DIR / "table_5_9_deberta_multisource.csv", index=False)
    print(table59.to_string(index=False))

In [ ]:
# =========================
# 12. 选最优实验并导出所有实验汇总
# =========================
metric_files = list(OUTPUT_DIR.glob("*/metrics_summary.csv"))
all_metrics = []
for f in metric_files:
    all_metrics.append(pd.read_csv(f))
all_metrics_df = pd.concat(all_metrics, axis=0).reset_index(drop=True)
all_metrics_df = all_metrics_df.sort_values("roc_auc", ascending=False).reset_index(drop=True)
all_metrics_df.to_csv(OUTPUT_DIR / "all_experiment_metrics.csv", index=False)

best_exp_name = all_metrics_df.iloc[0]["exp_name"]
best_exp_dir = OUTPUT_DIR / best_exp_name

print("BEST EXPERIMENT =", best_exp_name)
print(all_metrics_df.head(10).to_string(index=False))

In [ ]:
# =========================
# 14. 表5-13：数据集统计特征
# =========================
table513 = pd.DataFrame([
    dataset_stats("官方训练集", official_data),
    dataset_stats("官方验证集", valid_official),
    dataset_stats("外部数据集A", external_A),
    dataset_stats("外部数据集B", external_B),
])
table513.to_csv(OUTPUT_DIR / "table_5_13_dataset_stats.csv", index=False)
print(table513.to_string(index=False))

In [ ]:
# =========================
# 15. 图5-5 / 图5-6：长度分布与词汇统计
# =========================
if RUN_GROUPS["vis"]:
    plot_df = pd.concat([official_data, external_A, external_B], axis=0).reset_index(drop=True)
    plot_df["text_len"] = plot_df["text"].astype(str).str.len()
    plot_df["word_cnt"] = plot_df["text"].astype(str).str.split().apply(len)
    plot_df["ttr"] = plot_df["text"].apply(type_token_ratio)

    plt.figure(figsize=(8, 5))
    plt.hist(plot_df[plot_df["label"] == 0]["text_len"], bins=50, alpha=0.6, label="Human")
    plt.hist(plot_df[plot_df["label"] == 1]["text_len"], bins=50, alpha=0.6, label="AI")
    plt.xlabel("Text Length (chars)")
    plt.ylabel("Count")
    plt.title("Text Length Distribution")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_5_5_text_length_distribution.png", dpi=200)
    plt.show()

    avg_word_df = plot_df.groupby("label")["word_cnt"].mean().reset_index()
    avg_word_df["label_name"] = avg_word_df["label"].map({0: "Human", 1: "AI"})

    plt.figure(figsize=(6, 5))
    plt.bar(avg_word_df["label_name"], avg_word_df["word_cnt"])
    plt.xlabel("Class")
    plt.ylabel("Average Word Count")
    plt.title("Average Word Count by Class")
    plt.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_5_6_avg_word_count.png", dpi=200)
    plt.show()

    ttr_df = plot_df.groupby("label")["ttr"].agg(["mean", "std", "median"]).reset_index()
    ttr_df["label_name"] = ttr_df["label"].map({0: "Human", 1: "AI"})
    ttr_df.to_csv(OUTPUT_DIR / "lexical_ttr_stats.csv", index=False)
    print(ttr_df.to_string(index=False))

In [ ]:
# =========================
# 16. 图5-7：传统特征 t-SNE
# =========================
def sample_for_tsne(df, max_samples=MAX_TSNE_SAMPLES, seed=SEED):
    d0 = df[df["label"] == 0]
    d1 = df[df["label"] == 1]
    n = min(len(d0), len(d1), max_samples // 2)
    out = pd.concat([
        d0.sample(n=n, random_state=seed),
        d1.sample(n=n, random_state=seed)
    ], axis=0).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

def simple_text_features(df):
    feats = pd.DataFrame()
    feats["len_chars"] = df["text"].astype(str).str.len()
    feats["len_words"] = df["text"].astype(str).str.split().apply(len)
    feats["avg_word_len"] = df["text"].astype(str).apply(
        lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0
    )
    feats["punct_count"] = df["text"].astype(str).str.count(r"[,.!?;:]")
    feats["digit_count"] = df["text"].astype(str).str.count(r"\d")
    feats["upper_count"] = df["text"].astype(str).str.count(r"[A-Z]")
    return feats.values

def plot_tsne(X, y, title, save_path):
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, init="pca")
    X2 = tsne.fit_transform(X)

    plt.figure(figsize=(7, 6))
    plt.scatter(X2[y == 0, 0], X2[y == 0, 1], s=12, alpha=0.6, label="Human")
    plt.scatter(X2[y == 1, 0], X2[y == 1, 1], s=12, alpha=0.6, label="AI")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

if RUN_GROUPS["vis"]:
    tsne_df = sample_for_tsne(pd.concat([official_data, external_A, external_B], axis=0).reset_index(drop=True))
    X_simple = simple_text_features(tsne_df)
    y_simple = tsne_df["label"].values
    plot_tsne(X_simple, y_simple, "t-SNE of Simple Text Features", OUTPUT_DIR / "fig_5_7_tsne_simple_features.png")

In [ ]:
# =========================
# 17. 图5-8：DeBERTa 最后隐藏层 t-SNE
# =========================
def extract_deberta_embeddings(texts, model_dir, max_length=MAX_LENGTH_DEFAULT, batch_size=16):
    model = AutoModel.from_pretrained(str(model_dir)).to(DEVICE)
    tok = AutoTokenizer.from_pretrained(str(model_dir), use_fast=False)
    model.eval()

    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            enc = tok(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(DEVICE)

            outputs = model(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
            all_embs.append(cls_emb)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(all_embs, axis=0)

if RUN_GROUPS["vis"]:
    tsne_df2 = sample_for_tsne(pd.concat([official_data, external_A, external_B], axis=0).reset_index(drop=True), max_samples=1000 if QUICK_MODE else 1200)
    X_emb = extract_deberta_embeddings(
        tsne_df2["text"].tolist(),
        model_dir=best_exp_dir / "model",
        max_length=384,
        batch_size=16
    )
    y_emb = tsne_df2["label"].values
    plot_tsne(X_emb, y_emb, "t-SNE of DeBERTa Embeddings", OUTPUT_DIR / "fig_5_8_tsne_deberta_embeddings.png")

In [ ]:
# =========================
# 18. 表5-14：错误案例分析
# =========================
if RUN_GROUPS["error"]:
    pred_df = pd.read_csv(best_exp_dir / "valid_predictions.csv")
    valid_df = valid_official.copy().reset_index(drop=True).iloc[:len(pred_df)].copy()

    err_df = valid_df.copy()
    err_df["prob_ai"] = pred_df["prob_ai"].values
    err_df["pred"] = pred_df["pred"].values
    err_df["correct"] = (err_df["label"] == err_df["pred"]).astype(int)

    errors = err_df[err_df["correct"] == 0].copy()
    errors["真实标签"] = errors["label"].map({0: "Human", 1: "AI"})
    errors["预测标签"] = errors["pred"].map({0: "Human", 1: "AI"})
    errors["文本片段"] = errors["text"].astype(str).str.slice(0, 180)

    def reason_analysis(row):
        txt = str(row["text"])
        wc = len(txt.split())
        if row["label"] == 0 and row["pred"] == 1:
            if wc > 200:
                return "长文本结构规整，模型误判为AI"
            return "文本模板化明显"
        if row["label"] == 1 and row["pred"] == 0:
            if wc < 80:
                return "短文本信息不足，语义特征不明显"
            return "人工化改写后接近人类表达"
        return "边界样本"

    errors["误判原因分析"] = errors.apply(reason_analysis, axis=1)

    table514 = errors[["文本片段", "真实标签", "预测标签", "误判原因分析"]].head(20).copy()
    table514.insert(0, "编号", [f"C{i+1}" for i in range(len(table514))])
    table514["最优模型"] = "DeBERTa-v3-small"
    table514 = table514[["编号", "文本片段", "真实标签", "预测标签", "最优模型", "误判原因分析"]]

    table514.to_csv(OUTPUT_DIR / "table_5_14_error_cases.csv", index=False)
    print(table514.head(10).to_string(index=False))

In [ ]:
# =========================
# 19. 表5-15 + 图5-9：不同文本长度区间鲁棒性
# =========================
if RUN_GROUPS["error"]:
    pred_df = pd.read_csv(best_exp_dir / "valid_predictions.csv")
    valid_df = valid_official.copy().reset_index(drop=True).iloc[:len(pred_df)].copy()

    bucket_df = valid_df.copy()
    bucket_df["prob_ai"] = pred_df["prob_ai"].values
    bucket_df["pred"] = pred_df["pred"].values
    bucket_df["bucket"] = bucket_df["text"].apply(length_bucket_by_words)

    rows = []
    for b in ["0-100", "101-200", "201-400", "401+"]:
        sub = bucket_df[bucket_df["bucket"] == b]
        if len(sub) < 10 or sub["label"].nunique() < 2:
            rows.append({
                "文本长度区间": b,
                "样本数": len(sub),
                "DeBERTa AUC": np.nan,
                "DeBERTa Accuracy": np.nan,
                "DeBERTa F1": np.nan
            })
            continue

        auc_val = roc_auc_score(sub["label"], sub["prob_ai"])
        acc_val = accuracy_score(sub["label"], sub["pred"])
        _, _, f1_val, _ = precision_recall_fscore_support(sub["label"], sub["pred"], average="binary", zero_division=0)

        rows.append({
            "文本长度区间": b,
            "样本数": len(sub),
            "DeBERTa AUC": round(auc_val, 6),
            "DeBERTa Accuracy": round(acc_val, 6),
            "DeBERTa F1": round(f1_val, 6)
        })

    table515 = pd.DataFrame(rows)
    table515.to_csv(OUTPUT_DIR / "table_5_15_length_bucket_robustness.csv", index=False)
    print(table515.to_string(index=False))

    plt.figure(figsize=(8, 5))
    plt.bar(table515["文本长度区间"], table515["DeBERTa AUC"])
    plt.xlabel("Length Bucket")
    plt.ylabel("ROC-AUC")
    plt.title("Performance Across Length Buckets")
    plt.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_5_9_length_bucket_auc.png", dpi=200)
    plt.show()

In [ ]:
# =========================
# 20. 最终汇总
# =========================
summary = {
    "output_dir": str(OUTPUT_DIR.resolve()),
    "best_experiment": str(best_exp_name),
    "generated_files": [str(p) for p in sorted(OUTPUT_DIR.glob("**/*")) if p.is_file()]
}

with open(OUTPUT_DIR / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("全部完成，输出目录：", OUTPUT_DIR)
print("最优实验：", best_exp_name)